In [12]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

path = Path(
    r'D:\Project\Mini-Rag\data\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf'
)

loader = PyPDFLoader(str(path))
pages = loader.load()

print(len(pages))

print(pages[0].metadata)
print(pages[0].page_content[:1000])

564
{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-05-16T09:54:54+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'D:\\Project\\Mini-Rag\\data\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'total_pages': 564, 'page': 0, 'page_label': 'Cover'}
Aurélien Géron
Hands-On  
Machine Learning  
with Scikit-Learn  
& TensorFlow  
CONCEPTS, TOOLS, AND TECHNIQUES  
TO BUILD INTELLIGENT SYSTEMS
Download from finelybook www.finelybook.com


In [13]:
import unicodedata
import re

def clean_text(t: str) -> str:
    t = unicodedata.normalize('NFKC', t)
    t = t.encode('utf-8', errors='ignore').decode('utf-8')
    t = t.replace("\x00", "")               # حروف null بتيجي من بعض الـ PDFs
    t = re.sub(r"[ \t]+", " ", t)           # مسافات متكررة -> مسافة واحدة
    t = re.sub(r"\n{3,}", "\n\n", t)        # أسطر فاضية كتير -> سطرين
    return t.strip()

for d in pages:
    d.page_content = clean_text(d.page_content)

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

print(len(chunks))

print(chunks[0].metadata)

print(chunks[0].page_content)

1459
{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-03-10T21:55:34+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-05-16T09:54:54+08:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'D:\\Project\\Mini-Rag\\data\\Hands On Machine Learning with Scikit Learn and TensorFlow.pdf', 'total_pages': 564, 'page': 0, 'page_label': 'Cover', 'start_index': 0}
Aurélien Géron
Hands-On 
Machine Learning 
with Scikit-Learn 
& TensorFlow 
CONCEPTS, TOOLS, AND TECHNIQUES 
TO BUILD INTELLIGENT SYSTEMS
Download from finelybook www.finelybook.com


In [15]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

persist_dir = Path(r"D:\Project\Mini-Rag\Hand_on\Chroma_db_v1")

vector_store = Chroma(
    collection_name="hands_on_ml_book_clean",
    embedding_function=embedding_model,
    persist_directory=str(persist_dir),
)

BATCH_SIZE = 20
total_chunks = len(chunks)

for start in range(0, total_chunks, BATCH_SIZE):
    batch = [d for d in chunks[start : start + BATCH_SIZE]]
    vector_store.add_documents(batch)

    print(
        f"Stored through {min(start + len(batch), total_chunks)} "
        f"/ {total_chunks}"
    )

print("Final count:", vector_store._collection.count())


Stored through 20 / 1459
Stored through 40 / 1459
Stored through 60 / 1459
Stored through 80 / 1459
Stored through 100 / 1459
Stored through 120 / 1459
Stored through 140 / 1459
Stored through 160 / 1459
Stored through 180 / 1459
Stored through 200 / 1459
Stored through 220 / 1459
Stored through 240 / 1459
Stored through 260 / 1459
Stored through 280 / 1459
Stored through 300 / 1459
Stored through 320 / 1459
Stored through 340 / 1459
Stored through 360 / 1459
Stored through 380 / 1459
Stored through 400 / 1459
Stored through 420 / 1459
Stored through 440 / 1459
Stored through 460 / 1459
Stored through 480 / 1459
Stored through 500 / 1459
Stored through 520 / 1459
Stored through 540 / 1459
Stored through 560 / 1459
Stored through 580 / 1459
Stored through 600 / 1459
Stored through 620 / 1459
Stored through 640 / 1459
Stored through 660 / 1459
Stored through 680 / 1459
Stored through 700 / 1459
Stored through 720 / 1459
Stored through 740 / 1459
Stored through 760 / 1459
Stored through 7

In [16]:
results = vector_store.similarity_search_with_score(
    "What 's the Machine Learning?",
    k=3
)

for i, (doc, distance) in enumerate(results):
    print(f'Number: {i+1}')
    print(f'Score: {round(float(distance), 4)}')
    print(f'Page: {(doc.metadata.get('page_label'), doc.metadata.get('page'))}')
    print(f'Answer: \n{doc.page_content}')

Number: 1
Score: 0.3209
Page: ('4', 25)
Answer: 
If you already know all the Machine Learning basics, you may want
to skip directly to Chapter 2 . If you are not sure, tr
y to answer all
the questions listed at the end of the chapter before moving on.
What Is Machine Learning?
Machine Learning is the science (and art) of programming computers so they can
learn from data.
Here is a slightly more general definition:
[Machine Learning is the] field of study that gives computers the ability to learn
without being explicitly programmed.
—Arthur Samuel, 1959
And a more engineering-oriented one:
A computer program is said to learn from experience E with respect to some task T
and some performance measure P , if its performance on T, as measured by P , improves
with experience E.
—Tom Mitchell, 1997
For example, your spam filter is a Machine Learning program that can learn to flag
spam given examples of spam emails (e.g., flagged by users) and examples of regular
Number: 2
Score: 0.4549
Page: 

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model='qwen2.5-coder:7b',
    temperature=0
)

retriever = vector_store.as_retriever(
    search_kwargs = {'k':5}
)

prompt = ChatPromptTemplate.from_template("""
You are an assistant answering questions about a machine learning book.

Use ONLY the context below. Do NOT add any information, technique, or term that is not explicitly stated in the context.
If the context only partially answers the question, answer only that part.
If the context does not contain the answer, say:
"I don't know based on the provided book context."

Keep the answer clear and educational.

Context:
{context}

Question:
{question}

Answer:
""")

In [18]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f'[Page {doc.metadata.get("page_label", doc.metadata.get("page", 'Unknown'))}\n]'
        f'{doc.page_content}'
        for doc in docs
    )

rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
)

In [19]:
question = "What is Machine learning?"

answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:\n", answer)

Question: What is Machine learning?

Answer:
 Machine Learning is the science (and art) of programming computers so they can learn from data.


In [20]:
test_data = [
    # ---------- Ch1: The Machine Learning Landscape ----------
    {"id": 1, "chapter": 1,
     "question": "What is Machine Learning?",
     "expected": "The science of programming computers so they can learn from data without being explicitly programmed. A program learns from experience E with respect to task T and performance measure P if its performance on T, measured by P, improves with E."},
    {"id": 2, "chapter": 1,
     "question": "What is the difference between supervised and unsupervised learning?",
     "expected": "In supervised learning the training data includes the desired solutions (labels). In unsupervised learning the training data is unlabeled and the system tries to learn without a teacher."},
    {"id": 3, "chapter": 1,
     "question": "What is the difference between batch learning and online learning?",
     "expected": "Batch learning is trained offline using all available data and cannot learn incrementally. Online learning is trained incrementally by feeding data instances sequentially, individually or in mini-batches."},
    {"id": 4, "chapter": 1,
     "question": "What is overfitting and what are possible solutions?",
     "expected": "The model performs well on the training data but does not generalize well. Solutions: simplify the model (fewer parameters or regularization), gather more training data, reduce noise in the training data."},
    {"id": 5, "chapter": 1,
     "question": "Why do we need a test set, and what is a validation set used for?",
     "expected": "The test set estimates the generalization error on new instances. A validation set is used to compare models and select the best one and tune hyperparameters, so the test set stays untouched."},

    # ---------- Ch2: End-to-End Machine Learning Project ----------
    {"id": 6, "chapter": 2,
     "question": "What is the difference between RMSE and MAE?",
     "expected": "RMSE is the root of the mean squared error (norm l2) and is more sensitive to outliers. MAE is the mean absolute error (norm l1) and is preferred when there are many outliers."},
    {"id": 7, "chapter": 2,
     "question": "Why use stratified sampling when creating a test set?",
     "expected": "To make sure the test set is representative of the overall population, by sampling from homogeneous subgroups (strata) in the right proportions and avoiding sampling bias."},
    {"id": 8, "chapter": 2,
     "question": "What are the two common ways to do feature scaling?",
     "expected": "Min-max scaling (normalization), which rescales values to the range 0 to 1, and standardization, which subtracts the mean and divides by the standard deviation to get unit variance."},

    # ---------- Ch3: Classification ----------
    {"id": 9, "chapter": 3,
     "question": "What are precision and recall, and how are they computed from the confusion matrix?",
     "expected": "Precision = TP / (TP + FP), the accuracy of the positive predictions. Recall (sensitivity, true positive rate) = TP / (TP + FN), the ratio of positive instances correctly detected."},
    {"id": 10, "chapter": 3,
     "question": "What is the ROC curve and what does the AUC tell you?",
     "expected": "The ROC curve plots the true positive rate (recall) against the false positive rate. A perfect classifier has ROC AUC equal to 1, while a purely random classifier has 0.5."},

    # ---------- Ch4: Training Models ----------
    {"id": 11, "chapter": 4,
     "question": "What is the difference between the Normal Equation and Gradient Descent for Linear Regression?",
     "expected": "The Normal Equation computes the optimal parameters directly in closed form but gets slow with many features. Gradient Descent iteratively tweaks parameters to minimize the cost function and scales better with many features, but needs a learning rate and feature scaling."},
    {"id": 12, "chapter": 4,
     "question": "What are Batch, Stochastic and Mini-batch Gradient Descent?",
     "expected": "Batch GD uses the whole training set at each step. Stochastic GD picks one random instance at each step. Mini-batch GD uses small random subsets of instances at each step."},
    {"id": 13, "chapter": 4,
     "question": "What is the difference between Ridge Regression and Lasso Regression?",
     "expected": "Both are regularized linear models. Ridge adds an l2 penalty term, while Lasso adds an l1 penalty term and tends to eliminate the weights of the least important features, performing feature selection."},

    # ---------- Ch5: Support Vector Machines ----------
    {"id": 14, "chapter": 5,
     "question": "What is large margin classification in SVM and what are support vectors?",
     "expected": "SVM fits the widest possible street between the classes. The decision boundary is determined by the instances located on the edge of the street, called support vectors."},

    # ---------- Ch6: Decision Trees ----------
    {"id": 15, "chapter": 6,
     "question": "What is Gini impurity?",
     "expected": "A node is pure (gini = 0) if all training instances it applies to belong to the same class. Gini impurity measures how mixed the classes are in a node, computed as 1 minus the sum of squared class ratios."},

    # ---------- Ch7: Ensemble Learning and Random Forests ----------
    {"id": 16, "chapter": 7,
     "question": "What is the difference between hard voting and soft voting classifiers?",
     "expected": "Hard voting predicts the class that gets the majority of votes. Soft voting predicts the class with the highest class probability averaged over all classifiers, giving more weight to highly confident votes."},
    {"id": 17, "chapter": 7,
     "question": "What is the difference between bagging and pasting?",
     "expected": "Both train the same algorithm on different random subsets of the training set. Bagging samples with replacement, while pasting samples without replacement."},

    # ---------- Ch8: Dimensionality Reduction ----------
    {"id": 18, "chapter": 8,
     "question": "What is PCA and what is the explained variance ratio?",
     "expected": "Principal Component Analysis identifies the hyperplane closest to the data and projects the data onto it, preserving the maximum variance. The explained variance ratio indicates the proportion of the dataset's variance along each principal component."},

    # ---------- Ch11: Training Deep Neural Nets ----------
    {"id": 19, "chapter": 11,
     "question": "What is the vanishing gradients problem and how can it be mitigated?",
     "expected": "Gradients get smaller and smaller as the algorithm progresses down to the lower layers, so lower layers' weights are barely updated. Mitigations: better initialization (Xavier and He), non-saturating activation functions like ReLU and its variants, and Batch Normalization."},
    {"id": 20, "chapter": 11,
     "question": "What is Batch Normalization and why is it used?",
     "expected": "A technique that zero-centers and normalizes the inputs of each layer, then scales and shifts them, using the mean and standard deviation of the current mini-batch. It reduces vanishing gradients and the need for careful initialization, and acts as a regularizer."},

    # ---------- Out of scope: لازم يرد "I don't know" ----------
    {"id": 21, "chapter": None,
     "question": "Who won the 2018 FIFA World Cup?",
     "expected": "I don't know based on the provided book context."},
    {"id": 22, "chapter": None,
     "question": "How do I fine-tune a GPT model with LoRA?",
      "expected": "I don't know based on the provided book context."},
]

In [21]:
import json

results = []

for item in test_data:
    q = item["question"]
    retrieved = retriever.invoke(q)
    answer = rag_chain.invoke(q)

    results.append({
        "id": item["id"],
        "question": q,
        "expected": item["expected"],
        "answer": answer,
        "pages": [d.metadata.get("page_label", d.metadata.get("page")) for d in retrieved],
    })

    print(f"\n[{item['id']}] {q}")
    print("Pages retrieved:", results[-1]["pages"])
    print("EXPECTED:", item["expected"])
    print("ANSWER  :", answer)
    print("-" * 80)

with open("rag_test_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)


[1] What is Machine Learning?
Pages retrieved: ['4', '3', '29', '4', '7']
EXPECTED: The science of programming computers so they can learn from data without being explicitly programmed. A program learns from experience E with respect to task T and performance measure P if its performance on T, measured by P, improves with E.
ANSWER  : Machine Learning is the science (and art) of programming computers so they can learn from data. It is the field of study that gives computers the ability to learn without being explicitly programmed.
--------------------------------------------------------------------------------

[2] What is the difference between supervised and unsupervised learning?
Pages retrieved: ['494', '8', '10', '36', '13']
EXPECTED: In supervised learning the training data includes the desired solutions (labels). In unsupervised learning the training data is unlabeled and the system tries to learn without a teacher.
ANSWER  : In supervised learning, the training data includes t